# GPT-2 Text Training in Google Colab

This notebook trains a GPT-2 model on a custom text dataset. It's adapted from `scripts/train_text.py`.

## Setup

1. Make sure your Google Colab runtime is set to GPU (Runtime > Change runtime type > GPU).
2. Clone your repository to your Colab environment, if you haven't already, and navigate into its directory:
   ```bash
   # !git clone <your_repo_url>
   # %cd <your_repo_name>
   ```
3. Ensure your `requirements.txt` is in the root of your repository.

In [ ]:
# Install dependencies
!pip install -r requirements.txt

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## Configuration

Adjust the `BASE_DRIVE_PATH`, `DATA_DIR_RELATIVE`, and `OUTPUT_DIR_RELATIVE` variables below to match your Google Drive folder structure.

For example, if your project is in `MyDrive/my_nlp_project/`, then:
- `BASE_DRIVE_PATH = "/content/drive/MyDrive/my_nlp_project/"`
- Your processed data should be in `MyDrive/my_nlp_project/data/processed/train.txt`
- Your models will be saved in `MyDrive/my_nlp_project/models/text_colab_output/`

In [ ]:
import os

# --- Configuration Start ---
# TODO: USER - Define the base path to your project folder on Google Drive
BASE_DRIVE_PATH = "/content/drive/MyDrive/your_project_repo/" # IMPORTANT: Change this to your project's path in Google Drive

# Relative paths from the base project path
DATA_DIR_RELATIVE = "data/processed"
OUTPUT_DIR_RELATIVE = "models/text_colab_output" # Saving to a new dir to avoid conflicts with local outputs
# --- Configuration End ---

# Construct full paths
DATA_DIR = os.path.join(BASE_DRIVE_PATH, DATA_DIR_RELATIVE)
OUTPUT_DIR = os.path.join(BASE_DRIVE_PATH, OUTPUT_DIR_RELATIVE)

# Log the paths to verify
print(f"Base Drive Path: {BASE_DRIVE_PATH}")
print(f"Data Directory: {DATA_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

# Create output directory in Google Drive if it doesn't exist
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"Created output directory: {OUTPUT_DIR}")
else:
    print(f"Output directory already exists: {OUTPUT_DIR}")

## Training Script

The following cells contain the training logic, adapted from `scripts/train_text.py`.

In [ ]:
import logging
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import load_dataset
from transformers.trainer_utils import get_last_checkpoint
import torch
import os # Already imported, but good practice for self-contained cell

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def load_and_prepare_data(data_dir, max_length=512):
    train_file = os.path.join(data_dir, "train.txt")
    if not os.path.exists(train_file):
        logger.error(f"Training file not found at {train_file}")
        raise FileNotFoundError(f"Training file not found at {train_file}")
    
    logger.info(f"Loading dataset from {train_file}")
    dataset = load_dataset("text", data_files={"train": train_file})
    # Split into train and validation (e.g., 90% train, 10% validation)
    dataset = dataset["train"].train_test_split(test_size=0.1, seed=42) # Added seed for reproducibility
    
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token # Set pad_token to eos_token for GPT-2
    
    logger.info(f"Tokenizing dataset with max_length={max_length}")
    def tokenize_function(examples):
        outputs = tokenizer(
            examples["text"], 
            truncation=True, 
            padding="max_length", # Pad to max_length
            max_length=max_length,
            return_tensors="pt" # Return PyTorch tensors
        )
        # For language modeling, labels are usually the same as input_ids
        outputs["labels"] = outputs["input_ids"].clone()
        return outputs
    
    tokenized_datasets = dataset.map(
        tokenize_function, 
        batched=True,
        remove_columns=["text"] # Remove original text column
    )
    return tokenized_datasets, tokenizer

def train_model(tokenized_datasets, tokenizer, output_dir):
    # Check for GPU availability
    device = "cuda" if torch.cuda.is_available() else "cpu"
    logger.info(f"Using device: {device}")

    # Ensure output directory exists (already done in config cell, but good practice)
    os.makedirs(output_dir, exist_ok=True)
    
    # Load pre-trained GPT-2 model
    model = GPT2LMHeadModel.from_pretrained("gpt2")
    # Resize token embeddings if new tokens were added to tokenizer (not done here, but good practice)
    model.resize_token_embeddings(len(tokenizer))
    model.to(device) # Move model to the specified device (GPU or CPU)
    
    # Try to find the last checkpoint in the output directory
    last_checkpoint = get_last_checkpoint(output_dir)
    if last_checkpoint:
        logger.info(f"Resuming training from checkpoint: {last_checkpoint}")
    else:
        logger.info("No checkpoint found, starting training from scratch.")

    # Define TrainingArguments
    training_args = TrainingArguments(
        output_dir=output_dir, # Directory to save model checkpoints and logs
        learning_rate=5e-5,
        per_device_train_batch_size=4, # Adjust based on Colab GPU memory (e.g., T4 often needs 2 or 4)
        per_device_eval_batch_size=4, # Batch size for evaluation
        num_train_epochs=3, # Number of training epochs
        weight_decay=0.01,
        logging_dir=os.path.join(output_dir, "logs_colab"), # Directory for logs within output_dir
        logging_steps=100, # Log every N steps
        save_strategy="epoch", # Save a checkpoint at the end of each epoch
        evaluation_strategy="epoch", # Evaluate at the end of each epoch
        save_total_limit=3, # Keep only the last N checkpoints
        load_best_model_at_end=True, # Load the best model (based on eval loss) at the end of training
        # report_to="tensorboard", # Optional: Uncomment to use TensorBoard
        # fp16=torch.cuda.is_available(), # Optional: Uncomment for mixed precision training if GPU supports it
    )
    
    logger.info("Training arguments initialized successfully!")
    
    # Initialize the Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"], # Use the test split as validation set
        tokenizer=tokenizer, # Pass tokenizer to save it with the model
    )
    
    logger.info("Starting training...")
    # Use resume_from_checkpoint=last_checkpoint (True if path, False if None)
    trainer.train(resume_from_checkpoint=last_checkpoint if last_checkpoint else None)
    
    logger.info(f"Saving final model to {output_dir}")
    trainer.save_model(output_dir) # Saves the model and tokenizer
    
    return model, trainer

logger.info("Helper functions `load_and_prepare_data` and `train_model` defined.")

## Run Training

Make sure you have executed the **Configuration** cell (to set `DATA_DIR` and `OUTPUT_DIR`) before running the cell below.

In [ ]:
# --- Main Execution Cell ---

if "DATA_DIR" not in globals() or "OUTPUT_DIR" not in globals():
    print("ERROR: DATA_DIR or OUTPUT_DIR not configured. Please run the 'Configuration' cell (Cell 6) first!")
elif not os.path.exists(os.path.join(DATA_DIR, "train.txt")):
    print(f"ERROR: Training data 'train.txt' not found in {DATA_DIR}. \nPlease ensure your data is correctly placed in Google Drive and the BASE_DRIVE_PATH is set correctly.")
else:
    print("User-configured paths seem okay. Starting training process...")
    print(f"Data will be loaded from: {DATA_DIR}")
    print(f"Model outputs will be saved to: {OUTPUT_DIR}")
    
    print("\nStep 1: Loading and preparing data...")
    # DATA_DIR is the full Google Drive path configured in Cell 6
    tokenized_datasets, tokenizer = load_and_prepare_data(DATA_DIR)
    print("Data loading and tokenization complete.")
    
    print("\nStep 2: Training the model...")
    # OUTPUT_DIR is the full Google Drive path configured in Cell 6
    model, trainer = train_model(tokenized_datasets, tokenizer, OUTPUT_DIR)
    print("Model training complete.")
    
    print(f"\nFinal model and tokenizer have been saved to {OUTPUT_DIR}")
    print(f"You can find logs in a 'logs_colab' subdirectory within {OUTPUT_DIR}.")

## Notes
- **Data Location**: Ensure your `train.txt` is located at `your_project_folder_on_drive/data/processed/train.txt` (where `your_project_folder_on_drive` matches `BASE_DRIVE_PATH` you set in the Configuration cell).
- **Batch Size**: If you encounter Out-of-Memory (OOM) errors on Colab, try reducing `per_device_train_batch_size` (and `per_device_eval_batch_size`) in the `TrainingArguments` within the `train_model` function cell.
- **Resuming Training**: The script will automatically try to resume from the latest checkpoint if one is found in the `OUTPUT_DIR` on your Google Drive.
- **Output**: Models, checkpoints, and logs will be saved to the specified `OUTPUT_DIR` on your Google Drive.